# 00 — Async API Fetching & Caching

**Skill focus:** Data Engineering (intern/co-op/new grad)

This notebook demonstrates production-grade API ingestion patterns:
- **aiohttp + asyncio** — concurrent HTTP requests
- **Semaphore(20)** — rate limiting to respect API fair use
- **Exponential backoff** — retry on 429/5xx (1s → 2s → 4s)
- **Cache-first** — idempotent pipeline; re-runs never re-fetch
- **Ingestion manifest** — per-record status, bytes, timestamp

In [16]:
import sys
from pathlib import Path

project_root = Path.cwd()
if (project_root / "src").exists():
    pass
elif (project_root.parent / "src").exists():
    project_root = project_root.parent
elif (project_root.parent.parent / "src").exists():
    project_root = project_root.parent.parent  # notebooks/engineering/

sys.path.insert(0, str(project_root))

from src.ingestion import AsyncPokeAPIFetcher
from src.constants import POKEAPI_ENDPOINTS
import asyncio

## 1. Configure fetcher and cache paths

- **Local:** uses `data/cache/` (default)
- **Databricks:** pass `cache_root="/dbfs/FileStore/pokedata/cache"`

In [17]:
cache_root = project_root / "data" / "cache" if (project_root / "data").exists() else None

fetcher = AsyncPokeAPIFetcher(
    cache_root=str(cache_root) if cache_root else None,
    concurrency=20,
    max_retries=3,
)

print(f"Cache root: {fetcher.cache_root}")
print(f"Manifest:   {fetcher.manifest_path}")
print(f"Run ID:     {fetcher.run_id}")

Cache root: /Users/anirudh/UIUC/go-mama-27/personal-projects/pokedata/data/cache
Manifest:   /Users/anirudh/UIUC/go-mama-27/personal-projects/pokedata/data/cache/manifest.json
Run ID:     36ac5f0a


## 2. Async fetch all endpoints

- Checks cache before each request (cache hit → skip)
- Semaphore caps 20 concurrent connections
- tqdm progress bar per endpoint

In [18]:
DEMO_ENDPOINTS = {
    "pokemon": POKEAPI_ENDPOINTS["pokemon"],
    "pokemon-species": POKEAPI_ENDPOINTS["pokemon-species"],
    "type": POKEAPI_ENDPOINTS["type"],
    "move": POKEAPI_ENDPOINTS["move"],
}

results = await fetcher.fetch_all(DEMO_ENDPOINTS)


Endpoint: pokemon | Expected: 1025 records


  pokemon: 100%|██████████| 1025/1025 [00:00<00:00, 61222.43it/s]


  fetched=0 cached=1025 failed=0

Endpoint: pokemon-species | Expected: 1025 records


  pokemon-species: 100%|██████████| 1025/1025 [00:00<00:00, 71298.58it/s]


  fetched=0 cached=1025 failed=0

Endpoint: type | Expected: 18 records


  type: 100%|██████████| 18/18 [00:00<00:00, 5734.28it/s]


  fetched=0 cached=18 failed=0

Endpoint: move | Expected: 920 records


  move: 100%|██████████| 920/920 [00:00<00:00, 2652.55it/s]

  fetched=0 cached=919 failed=1


## 3. Post-fetch validation

Assert file counts match expected totals per endpoint.

In [19]:
valid = fetcher.validate_cache(DEMO_ENDPOINTS)
print(f"\nValidation: {'PASS' if valid else 'WARN (check output above)'}")

  PASS pokemon: 1025 files (expected ~1025)
  PASS pokemon-species: 1025 files (expected ~1025)
  PASS type: 18 files (expected ~18)
  PASS move: 919 files (expected ~920)

Validation: PASS


## 4. Manifest summary

Total records, success/failure counts, and cache size.

In [20]:
summary = fetcher.manifest_summary()
for k, v in summary.items():
    print(f"  {k}: {v}")

  total_records: 2988
  successful: 2987
  failed: 1
  total_mb: 207.65


## Key findings

- **Cache-first:** Re-running the notebook uses cached files; no duplicate API calls
- **Rate limiting:** Semaphore(20) prevents overwhelming the API
- **Resilience:** Exponential backoff handles transient 429/5xx errors
- **Traceability:** Manifest records status, bytes, and timestamp per (endpoint, id)